In [1]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import pandas as pd
import datetime
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

In [7]:
df = pd.read_excel('urbox_bi_analyst_test.xlsx', sheet_name = 'Data for Part 1')


In [19]:
query = """with first_date as (select user_id, brand_id , min (voucher_redeemed_at) first_brand_date
from df
group by user_id, brand_id),

rank_table as (select user_id, brand_id, first_brand_date,
row_number() over (partition by user_id order by first_brand_date asc) asc_rank,
row_number() over (partition by user_id order by first_brand_date desc) desc_rank, 
count(brand_id) over (partition by user_id) number_of_brands
from first_date)


select s.user_id, s.brand_id as first_brand_id, t.brand_id second_brand_id, s.first_brand_date first_brand_redeemed_date, 
t.first_brand_date second_brand_redeemed_date, k.brand_id last_brand_id, k.first_brand_date as last_brand_redeemed_date, s.number_of_brands
from rank_table as s
left join rank_table  as t on s.user_id = t.user_id and t.asc_rank = 2 and s.brand_id != t.brand_id
left join rank_table as k on s.user_id = k.user_id and k.desc_rank = 1 and s.brand_id != k.brand_id and t.brand_id!= k.brand_id
where s.asc_rank = 1

"""

In [21]:
result = duckdb.query(query).to_df()
print(result)

        user_id  first_brand_id  second_brand_id first_brand_redeemed_date  \
0       1571771             910             <NA>       2025-06-26 21:36:50   
1      20468098             661             <NA>       2025-04-20 01:24:00   
2    1000013661             395             <NA>       2025-01-04 03:31:18   
3    1000207820            1110             <NA>       2025-01-07 01:41:50   
4    1000253298             274             <NA>       2025-01-09 04:10:25   
..          ...             ...              ...                       ...   
995  1007346345             274               82       2025-05-05 04:09:12   
996  1008646057             593              552       2025-01-01 16:56:21   
997  1008713941             516              138       2025-01-08 19:11:53   
998  1014541110            1511               82       2025-03-27 17:50:30   
999  1014864019             273              766       2025-03-06 00:40:36   

    second_brand_redeemed_date  last_brand_id last_brand_redeem